In [1]:
import re
import requests
from PIL import Image
from io import BytesIO

from transformers import DonutProcessor, VisionEncoderDecoderModel
from datasets import load_dataset
import torch

/opt/homebrew/Caskroom/miniforge/base/envs/ml-cli/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
processor = DonutProcessor.from_pretrained("naver-clova-ix/donut-base-finetuned-cord-v2")
model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base-finetuned-cord-v2")

image_prefix_url = "https://prices.openfoodfacts.org/img/"

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
# load document image
dataset = load_dataset("hf-internal-testing/example-documents", split="test")
dataset_ofp = load_dataset("openfoodfacts/open-prices")['prices']
image = dataset[2]["image"]

Config of the encoder: <class 'transformers.models.donut.modeling_donut_swin.DonutSwinModel'> is overwritten by shared encoder config: DonutSwinConfig {
  "attention_probs_dropout_prob": 0.0,
  "depths": [
    2,
    2,
    14,
    2
  ],
  "drop_path_rate": 0.1,
  "embed_dim": 128,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 1024,
  "image_size": [
    1280,
    960
  ],
  "initializer_range": 0.02,
  "layer_norm_eps": 1e-05,
  "mlp_ratio": 4.0,
  "model_type": "donut-swin",
  "num_channels": 3,
  "num_heads": [
    4,
    8,
    16,
    32
  ],
  "num_layers": 4,
  "patch_size": 4,
  "path_norm": true,
  "qkv_bias": true,
  "transformers_version": "4.46.3",
  "use_absolute_embeddings": false,
  "window_size": 10
}

Config of the decoder: <class 'transformers.models.mbart.modeling_mbart.MBartForCausalLM'> is overwritten by shared decoder config: MBartConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_cross_attention": true,
  "add_

In [3]:
image.show()

In [4]:
dataset_ofp = dataset_ofp.filter(lambda x: x['proof_type'] == 'RECEIPT')
dataset_ofp

Filter: 100%|██████████| 57033/57033 [00:02<00:00, 23812.66 examples/s]


Dataset({
    features: ['id', 'type', 'product_code', 'product_name', 'category_tag', 'labels_tags', 'origins_tags', 'price', 'price_is_discounted', 'price_without_discount', 'price_per', 'currency', 'location_osm_id', 'location_osm_type', 'location_id', 'date', 'proof_id', 'receipt_quantity', 'owner', 'source', 'created', 'updated', 'proof_file_path', 'proof_mimetype', 'proof_type', 'proof_date', 'proof_currency', 'proof_receipt_price_count', 'proof_receipt_price_total', 'proof_owner', 'proof_source', 'proof_created', 'proof_updated', 'location_type', 'location_osm_display_name', 'location_osm_tag_key', 'location_osm_tag_value', 'location_osm_address_postcode', 'location_osm_address_city', 'location_osm_address_country', 'location_osm_address_country_code', 'location_osm_lat', 'location_osm_lon', 'location_website_url', 'location_source', 'location_created', 'location_updated'],
    num_rows: 10008
})

In [5]:
image_url = image_prefix_url+dataset_ofp[0]['proof_file_path']
response = requests.get(image_url)
response.raise_for_status()
image = Image.open(BytesIO(response.content))
# Open the image
# image = Image.open(BytesIO(response.content))
#
# # Display the image (optional)
# image.show()
# image_url
image.show()

In [6]:
# prepare decoder inputs
task_prompt = "<s_cord-v2>"
decoder_input_ids = processor.tokenizer(task_prompt, add_special_tokens=False, return_tensors="pt").input_ids

pixel_values = processor(image, return_tensors="pt").pixel_values

outputs = model.generate(
    pixel_values.to(device),
    decoder_input_ids=decoder_input_ids.to(device),
    max_length=model.decoder.config.max_position_embeddings,
    pad_token_id=processor.tokenizer.pad_token_id,
    eos_token_id=processor.tokenizer.eos_token_id,
    use_cache=True,
    bad_words_ids=[[processor.tokenizer.unk_token_id]],
    return_dict_in_generate=True,
)

sequence = processor.batch_decode(outputs.sequences)[0]
sequence = sequence.replace(processor.tokenizer.eos_token, "").replace(processor.tokenizer.pad_token, "")
sequence = re.sub(r"<.*?>", "", sequence, count=1).strip()  # remove first task start token
print(processor.token2json(sequence))

{'menu': [{'nm': 'SILINGRAD', 'unitprice': '76. 43.66.70', 'cnt': '04.', 'price': '76.43.66.70', 'sub': [{'nm': 'Volkswagen. fr'}, {'nm': 'Vous avez ete accueilli par SCO'}]}, {'nm': {'cnt': '133', 'price': '63700'}, 'unitprice': '18: 19: 59', 'cnt': 'a'}], 'sub_total': {'subtotal_price': '4,59', 'price': '3,13', 'tax_price': '4,59', 'etc': '3,13'}, 'total': {'total_price': '4,59', 'cashprice': '4,35', 'changeprice': '4,59'}}
